In [1]:
import os
import sys
import json
import time
import requests
from typing import Any, Dict, Optional, Tuple
from pymongo import MongoClient
import random

# =========================
# Config via env vars
# =========================
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB")
COLLECTION_NAME = os.getenv("MONGO_COLLECTION")

SHOP =  "71eaf7.myshopify.com"
TOKEN = os.getenv("SHOPIFY_ADMIN_ACCESS_TOKEN")
API_VERSION = os.getenv("SHOPIFY_API_VERSION", "2025-10")

# 前台商品链接用这个域名拼，建议填你的真实网站域名：
# e.g. "https://luxuryevermore.com"
STOREFRONT_BASE = os.getenv("SHOPIFY_STOREFRONT_BASE_URL")

# 轮询间隔（避免太快）
SLEEP_SECONDS = float(os.getenv("SLEEP_SECONDS", "0.15"))

# 可选：只处理状态为某些值的 item（留空则处理全部）
STATUS_FILTER = os.getenv("STATUS_FILTER", "").strip()  # e.g. "available,listed"

SESSION = requests.Session()

# =========================
# Shopify GraphQL
# =========================
QUERY_MIN_FOR_MONGO = """
query MinForMongo($q: String!) {
  productVariants(first: 2, query: $q) {
    edges {
      node {
        id
        sku
        price
        compareAtPrice
        inventoryQuantity

        product {
          title
          handle
          featuredImage { url }
        }
      }
    }
  }
}
"""


def shopify_gql(query: str, variables: Dict[str, Any]) -> Dict[str, Any]:
    if not SHOP or not TOKEN:
        raise RuntimeError("Missing env vars SHOPIFY_SHOP / SHOPIFY_ADMIN_ACCESS_TOKEN")

    url = f"https://{SHOP}/admin/api/{API_VERSION}/graphql.json"
    headers = {
        "Content-Type": "application/json",
        "X-Shopify-Access-Token": TOKEN,
    }

    max_attempts = 3
    for attempt in range(1, max_attempts + 1):
        try:
            resp = SESSION.post(
                url,
                headers=headers,
                json={"query": query, "variables": variables},
                timeout=30,
            )

            # 429 限流：按 Retry-After 等待后重试（算作一次 attempt）
            if resp.status_code == 429:
                retry_after = resp.headers.get("Retry-After")
                wait_s = float(retry_after) if retry_after else 2.0
                time.sleep(wait_s)
                continue

            resp.raise_for_status()
            payload = resp.json()

            if "errors" in payload and payload["errors"]:
                raise RuntimeError("GraphQL errors:\n" + json.dumps(payload["errors"], ensure_ascii=False, indent=2))

            return payload["data"]

        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
            # 网络抖动：重试几次，最后一次仍失败则抛出（触发你现有的 sys.exit(1)）
            if attempt == max_attempts:
                raise
            backoff = 1.0 * attempt + random.random()  # 轻量退避
            time.sleep(backoff)
            continue


def build_product_url(handle: Optional[str]) -> Optional[str]:
    if not handle:
        return None

    # 你没填 STOREFRONT_BASE，就无法保证拼出来是可访问的前台域名
    # （myshopify.com 也能访问，但不一定是你的对外展示域名）
    if STOREFRONT_BASE:
        base = STOREFRONT_BASE.rstrip("/")
        return f"{base}/products/{handle}"

    # fallback：用 myshopify 域名拼（可能可用，但不建议当最终展示）
    return f"https://{SHOP}/products/{handle}"


def fetch_shopify_details_by_sku(sku: str) -> Dict[str, Any]:
    """
    Returns:
      { ok: true, details: {...} }
      or { ok: false, error: "SKU not found", sku: sku }
    Any other error -> raises RuntimeError (terminate program)
    """
    
    q = f"sku:{sku}"
    data = shopify_gql(QUERY_MIN_FOR_MONGO, {"q": q})

    edges = data["productVariants"]["edges"]
    if not edges:
        return {"ok": False, "error": "SKU not found", "sku": sku}
    if len(edges) > 1:
        raise RuntimeError(f"Duplicate SKU found in Shopify for sku={sku} (matches={len(edges)})")

    node = edges[0]["node"]
    product = node["product"] or {}

    handle = product.get("handle")
    featured_image = (product.get("featuredImage") or {}).get("url")

    details = {
        "url": build_product_url(handle),
        "title": product.get("title"),
        "price": node.get("price"),
        "compare_at_price": node.get("compareAtPrice"),
        "inventory_quantity": node.get("inventoryQuantity"),
        "featured_image": featured_image,
    }
    return {"ok": True, "details": details}


def main():
    client = MongoClient(MONGO_URI)
    col = client[DB_NAME][COLLECTION_NAME]

    query: Dict[str, Any] = {}

    # 统一排除 SOLD
    query["status"] = {"$ne": "SOLD"}

    # 如果你还想保留 STATUS_FILTER（可选），就让它叠加到同一个 status 条件里
    if STATUS_FILTER:
        allowed = [s.strip() for s in STATUS_FILTER.split(",") if s.strip()]
        if allowed:
            query["status"] = {"$in": [s for s in allowed if s != "SOLD"]}


    # 只取 sku + _id，减少 IO
    cursor = col.find(query, {"sku": 1, "_id": 1})

    processed = 0
    found = 0
    not_found = 0

    for doc in cursor:
        processed += 1
        sku = (doc.get("sku") or "").strip()
        if not sku:
            # 没有 sku 的记录：标记为不存在（你也可以选择跳过）
            col.update_one(
                {"_id": doc["_id"]},
                {
                    "$set": {"shopify_sku_exist": False},
                    "$unset": {"shopify_details": ""}   # 👈 完全删除字段
                },
            )
            continue

        try:
            res = fetch_shopify_details_by_sku(sku)
        except Exception as e:
            # 任何非 “SKU not found” 的错误都终止
            print(f"\n❌ Fatal error on SKU={sku}: {e}")
            sys.exit(1)

        if res.get("ok") is False:
            # 只允许这一种 error 继续跑
            if res.get("error") == "SKU not found":
                not_found += 1
                col.update_one(
                    {"_id": doc["_id"]},
                    {
                        "$set": {"shopify_sku_exist": False},
                        "$unset": {"shopify_details": ""}
                    },
                )

            else:
                print(f"\n❌ Fatal unexpected error on SKU={sku}: {res}")
                sys.exit(1)
        else:
            found += 1
            col.update_one(
                {"_id": doc["_id"]},
                {"$set": {"shopify_sku_exist": True, "shopify_details": res["details"]}},
            )

        if SLEEP_SECONDS > 0:
            time.sleep(SLEEP_SECONDS)

        # 简单进度输出
        if processed % 50 == 0:
            print(f"Processed={processed} Found={found} NotFound={not_found}")
    client.close()
    print(f"\n✅ Done. Processed={processed} Found={found} NotFound={not_found}")



RUN_INTERVAL_SECONDS = 10 * 60  # 每轮跑完休眠10分钟

if __name__ == "__main__":
    while True:
        try:
            main()
        except SystemExit:
            # 你的逻辑里遇到非“SKU not found”会 sys.exit(1)，这里保持原行为：直接退出整个循环/程序
            raise
        except Exception as e:
            # 如果你希望任何未捕获异常也终止，就直接 raise；
            # 如果你希望异常后仍然10分钟后重跑，就改成 print 后继续 sleep。
            print(f"\n❌ Unhandled fatal error: {e}")
            raise
        
        print(f"\n⏳ Sleeping {RUN_INTERVAL_SECONDS} seconds before next run...")
        time.sleep(RUN_INTERVAL_SECONDS)




❌ Unhandled fatal error: name must be an instance of str


TypeError: name must be an instance of str

NameError: name 'payload' is not defined